In [3]:
import tensorflow as tf

print(tf.__version__)

ValueError: The proto value '6' is already registered.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib
import os

print("All libraries imported successfully!")

In [ ]:
df = pd.read_csv("../data/raw/student-por.csv", sep=";")

print("Dataset Shape:", df.shape)

df.head()

In [ ]:
data = df.copy()

# Remove previous grades for early prediction
data = data.drop(columns=["G1", "G2"])

# Input features
X = data.drop(columns=["G3"])

# Target variable
y = data["G3"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

In [ ]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Records:", len(X_train))
print("Testing Records:", len(X_test))

In [ ]:
numerical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Training Shape:", X_train_processed.shape)
print("Testing Shape:", X_test_processed.shape)

In [ ]:
X_train_processed = np.asarray(
    X_train_processed,
    dtype=np.float32
)

X_test_processed = np.asarray(
    X_test_processed,
    dtype=np.float32
)

y_train = np.asarray(y_train, dtype=np.float32)

y_test = np.asarray(y_test, dtype=np.float32)

print("Data Conversion Completed!")

In [ ]:
model = keras.Sequential([

    keras.Input(shape=(X_train_processed.shape[1],)),

    layers.Dense(64, activation="relu"),

    layers.Dense(32, activation="relu"),

    layers.Dense(16, activation="relu"),

    layers.Dense(1, activation="linear")

])

model.summary()

In [ ]:
model.compile(

    optimizer="adam",

    loss="mean_squared_error",

    metrics=["mae"]

)

print("Model Compiled Successfully!")

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(

    monitor="val_loss",

    patience=20,

    restore_best_weights=True

)

In [ ]:
history = model.fit(

    X_train_processed,

    y_train,

    validation_split=0.2,

    epochs=200,

    batch_size=16,

    callbacks=[early_stopping],

    verbose=1

)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history.history["loss"], label="Training Loss")

plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("ANN Training and Validation Loss")

plt.xlabel("Epochs")

plt.ylabel("Mean Squared Error")

plt.legend()

plt.show()

In [ ]:
predictions = model.predict(X_test_processed)

predictions = predictions.flatten()

comparison = pd.DataFrame({

    "Actual": y_test,

    "Predicted": predictions

})

comparison.head(10)

In [ ]:
mae = mean_absolute_error(y_test, predictions)

mse = mean_squared_error(y_test, predictions)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, predictions)

print("ANN Model Evaluation")

print("MAE:", mae)

print("MSE:", mse)

print("RMSE:", rmse)

print("R2 Score:", r2)

In [ ]:
plt.figure(figsize=(7, 5))

plt.scatter(y_test, predictions, alpha=0.7)

plt.xlabel("Actual Grades")

plt.ylabel("Predicted Grades")

plt.title("ANN: Actual vs Predicted Grades")

plt.show()

In [ ]:
os.makedirs("../models", exist_ok=True)

model.save("../models/student_performance_ann.keras")

joblib.dump(
    preprocessor,
    "../models/ann_preprocessor.pkl"
)

print("ANN Model and Preprocessor Saved Successfully!")

In [ ]:
loaded_model = keras.models.load_model(
    "../models/student_performance_ann.keras"
)

loaded_preprocessor = joblib.load(
    "../models/ann_preprocessor.pkl"
)

print("Saved Model Loaded Successfully!")

In [ ]:
# Select one student from test data
sample_student = X_test.iloc[[0]]

# Transform student information
sample_processed = loaded_preprocessor.transform(sample_student)

sample_processed = np.asarray(
    sample_processed,
    dtype=np.float32
)

# Predict grade
predicted_grade = loaded_model.predict(sample_processed)

print("Predicted Final Grade:", predicted_grade[0][0])
print("Actual Final Grade:", y_test[0])